# 1. Data: loading, exploring, cleaning, merging

## 1.1 First look at the raw data

In [2]:
import pandas as pd
import os

cso = pd.read_csv("data/raw/CPM15.csv")
cso.head()

,STATISTIC,Statistic Label,TLIST(M1),Month,C01779V04002,Commodity Group,UNIT,VALUE
0,CPM15C01,EU HICP,199601,1996 January,-,All-items HICP (COICOP 00),Base 2015=100,68.1
1,CPM15C01,EU HICP,199601,1996 January,01,Food and non-alcoholic beverages (COICOP 01),Base 2015=100,79.0
2,CPM15C01,EU HICP,199601,1996 January,011,Food (COICOP 01.1),Base 2015=100,79.3
3,CPM15C01,EU HICP,199601,1996 January,0111,Bread and cereals (COICOP 01.1.1),Base 2015=100,75.5
4,CPM15C01,EU HICP,199601,1996 January,0112,Meat (COICOP 01.1.2),Base 2015=100,87.8


In [3]:
cso.shape

(244800, 8)

In [4]:
cso.info()

<class 'pandas.DataFrame'>
RangeIndex: 244800 entries, 0 to 244799
Data columns (total 8 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   STATISTIC        244800 non-null  str    
 1   Statistic Label  244800 non-null  str    
 2   TLIST(M1)        244800 non-null  int64  
 3   Month            244800 non-null  str    
 4   C01779V04002     244800 non-null  str    
 5   Commodity Group  244800 non-null  str    
 6   UNIT             244800 non-null  str    
 7   VALUE            224189 non-null  float64
dtypes: float64(1), int64(1), str(6)
memory usage: 14.9 MB


In [5]:
cso.isna().sum()

STATISTIC              0
Statistic Label        0
TLIST(M1)              0
Month                  0
C01779V04002           0
Commodity Group        0
UNIT                   0
VALUE              20611
dtype: int64

In [6]:
eu = pd.read_csv("data/raw/prc_hicp_manr.csv", nrows=1000)
eu.head()

,DATAFLOW,LAST UPDATE,freq,unit,coicop,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:PRC_HICP_MANR(1.0),06/02/26 23:00:00,Monthly,Annual rate of change,Administered prices,Albania,2019-12,0.0,d,NaN
1,ESTAT:PRC_HICP_MANR(1.0),06/02/26 23:00:00,Monthly,Annual rate of change,Administered prices,Albania,2020-01,0.1,d,NaN
2,ESTAT:PRC_HICP_MANR(1.0),06/02/26 23:00:00,Monthly,Annual rate of change,Administered prices,Albania,2020-02,0.0,d,NaN
3,ESTAT:PRC_HICP_MANR(1.0),06/02/26 23:00:00,Monthly,Annual rate of change,Administered prices,Albania,2020-03,0.0,d,NaN
4,ESTAT:PRC_HICP_MANR(1.0),06/02/26 23:00:00,Monthly,Annual rate of change,Administered prices,Albania,2020-04,0.0,d,NaN


In [7]:
eu.shape

(1000, 10)

In [8]:
eu.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DATAFLOW     1000 non-null   str    
 1   LAST UPDATE  1000 non-null   str    
 2   freq         1000 non-null   str    
 3   unit         1000 non-null   str    
 4   coicop       1000 non-null   str    
 5   geo          1000 non-null   str    
 6   TIME_PERIOD  1000 non-null   str    
 7   OBS_VALUE    1000 non-null   float64
 8   OBS_FLAG     73 non-null     str    
 9   CONF_STATUS  0 non-null      float64
dtypes: float64(2), str(8)
memory usage: 78.3 KB


In [9]:
eu.isna().sum()

DATAFLOW          0
LAST UPDATE       0
freq              0
unit              0
coicop            0
geo               0
TIME_PERIOD       0
OBS_VALUE         0
OBS_FLAG        927
CONF_STATUS    1000
dtype: int64

## 1.2 Cleaning the CSO data (CPM15)

In [10]:
cso = pd.read_csv("data/raw/CPM15.csv", dtype={"C01779V04002": str})
cso.shape

(244800, 8)

What: load the CSO file again, telling pandas to treat the category code column as text.

Why: codes like 01 and 04 start with a zero. If pandas thought they were numbers, it would turn 01 into 1 and the zero would be lost. Then the codes would stop matching the official COICOP codes.

## Step 1.2: Keep only the “annual rate of change”

In [11]:
cso = cso[cso["Statistic Label"] == "Annual rate of change in HICP"]
cso.shape

(48960, 8)

What: keep only the rows where the statistic is the annual rate of change.

Why: the CSO file has 5 different measures mixed together, and the Eurostat file only has one: the annual rate of change. To compare Ireland with Europe fairly, both must measure the same thing. Comparing an index (like 120) with a percentage (like 8%) would be like comparing kilos with metres.

Result: (48960, 8). That’s 244,800 ÷ 5, so exactly one fifth is left, which makes sense.

## Step 1.3: Keep only the 4 categories

In [12]:
cso = cso[cso["C01779V04002"].isin(["-", "01", "04", "ZZZ3"])]
cso.shape

(1440, 8)

What: keep only these four codes:

Code	Category
-	All items
01	Food and non-alcoholic beverages
04	Housing, water, electricity, gas and other fuels
ZZZ3	Energy

Why: these are the 4 categories you chose in Stage 1. The other 132 aren’t needed for your story, and keeping them would only make the data bigger and harder to work with. We filter on codes, not names, because codes are exact. Names can have small spelling differences.

Result: (1440, 8), which is 4 categories × 360 months.

## Step 1.4: Keep only January 2019 to December 2025

In [13]:
cso = cso[(cso["TLIST(M1)"] >= 201901) & (cso["TLIST(M1)"] <= 202512)]
cso.shape

(336, 8)

- **2019:** prices before the crisis (baseline)
- **2022:** the peak of the price surge
- **2025:** the cool-down, and the last month in both datasets

`TLIST(M1)` stores the month as a number in the form YYYYMM (e.g. `201901`), so a simple greater-than / less-than filter works.

**Expected result:** 336 rows (4 categories × 84 months).

## Step 1.5: Remove the columns you don’t need

In [14]:
cso = cso.drop(columns=["STATISTIC", "Statistic Label", "Month", "UNIT"])
cso.columns

Index(['TLIST(M1)', 'C01779V04002', 'Commodity Group', 'VALUE'], dtype='str')

| Column | Why it was removed |
|---|---|
| `STATISTIC` | After Step 2, every row has the same value (`CPM15C03`) |
| `Statistic Label` | After Step 2, every row says "Annual rate of change in HICP" |
| `Month` | Same date as `TLIST(M1)`, written in words; only one date column is needed |
| `UNIT` | After Step 2, every row says `%` |

A column with the same value in every row tells the reader nothing and only makes the table wider.

## Step 1.6: Rename the columns

In [15]:
cso = cso.rename(columns={
    "TLIST(M1)": "month",
    "C01779V04002": "coicop_code",
    "Commodity Group": "category_full",
    "VALUE": "value"
})
cso.head()

,month,coicop_code,category_full,value
135456,201901,-,All-items HICP (COICOP 00),0.8
135457,201901,01,Food and non-alcoholic beverages (COICOP 01),-1.6
135484,201901,04,"Housing, water, electricity, gas and other fue...",5.1
135587,201901,ZZZ3,Energy,1.6
135592,201902,-,All-items HICP (COICOP 00),0.7


What: give every column a short, clear name.

Why: C01779V04002 means nothing to a person. Clear names make the code easier to read, and Priyanka will understand the file without asking you. Lowercase names without spaces are also easier to type in code (month instead of "TLIST(M1)"). Later the Eurostat columns will get the same names, so the two tables can be joined.

## Step 1.7: Fix the “All items” code

In [16]:
cso["coicop_code"] = cso["coicop_code"].replace("-", "00")
cso["coicop_code"].unique()

<StringArray>
['00', '01', '04', 'ZZZ3']
Length: 4, dtype: str

What: change the code - to 00.

Why: in the official COICOP system, “All items” is code 00. The CSO file writes it as a dash -, but its own label even says “(COICOP 00)”. A dash looks like a missing value and could confuse anyone reading the data, so we use the correct official code.

Result: ['00', '01', '04', 'ZZZ3']

## Step 1.8: Turn the month into a real date

In [17]:
cso["month"] = pd.to_datetime(cso["month"].astype(str), format="%Y%m")
cso.dtypes

month            datetime64[us]
coicop_code                 str
category_full               str
value                   float64
dtype: object

What: change 201901 (a plain number) into a real date: 2019-01-01.

Why: to Python, 201901 is just a number, like 201,901. It doesn’t know it’s a month. A real date lets the chart put months in the correct order and space them properly, and it’s the only way to match the CSO dates with Eurostat’s 2019-12 style later.

.astype(str) turns the number into text first, because the date converter reads text.
format="%Y%m" tells it how to read the text: %Y = 4-digit year, %m = 2-digit month.

Result: month now shows as datetime64.

## Step 1.9: Add a short category name and the country

In [18]:
cso["category"] = cso["coicop_code"].map({
    "00": "All items",
    "01": "Food",
    "04": "Housing",
    "ZZZ3": "Energy"
})
cso["geo"] = "Ireland"
cso = cso.drop(columns=["category_full"])
cso.head()

,month,coicop_code,value,category,geo
135456,2019-01-01,00,0.8,All items,Ireland
135457,2019-01-01,01,-1.6,Food,Ireland
135484,2019-01-01,04,5.1,Housing,Ireland
135587,2019-01-01,ZZZ3,1.6,Energy,Ireland
135592,2019-02-01,00,0.7,All items,Ireland


- **Short names** (All items, Food, Housing, Energy): long names like "Housing, water, electricity, gas and other fuels (COICOP 04)" are too long for chart labels. The same short names will be given to the Eurostat data, so the two files match.
- **`geo` = Ireland:** the CSO file only covers Ireland, so it had no country column. Eurostat covers 45 places and needs one. Both tables need the same columns to be joined.
- **`category_full` removed:** the short name and the code already carry the same information.

## Step 1.10: Final check

In [19]:
cso = cso[["month", "geo", "coicop_code", "category", "value"]]
cso.shape

(336, 5)

In [20]:
cso.isna().sum()

month          0
geo            0
coicop_code    0
category       0
value          0
dtype: int64

In [21]:
cso.groupby("category").size()

category
All items    84
Energy       84
Food         84
Housing      84
dtype: int64

What: put the columns in a sensible order and check the result.

Why: always check after cleaning, so you can be sure nothing went wrong. You should see:

336 rows, 5 columns,
0 missing values in every column,
84 rows for each category, one per month.

## 1.3 Cleaning the Eurostat data (prc_hicp_manr)

**Goal:** keep only the 4 categories, the 27 EU countries plus the euro area, and Jan 2019 – Dec 2025, and give the table the same columns as the cleaned CSO table.
### Step 1: Load only the useful columns

In [22]:
eu = pd.read_csv("data/raw/prc_hicp_manr.csv",
                 usecols=["coicop", "geo", "TIME_PERIOD", "OBS_VALUE", "OBS_FLAG"])
eu.shape

(3533048, 5)

The file is 491.6 MB. Five of its 10 columns are not needed:
- `DATAFLOW`, `freq`, `unit`: the same value in every row
- `LAST UPDATE`: the same date in every row (recorded in the README instead)
- `CONF_STATUS`: completely empty

Loading only the 5 useful columns (`usecols`) makes the file load faster and use much less memory.

**Expected result:** 3,533,048 rows. That's 3.4× Excel's limit of 1,048,576 rows, which is why Python is needed.

## Step 2: Keep only the 4 chosen categories

In [23]:
categories = ["All-items HICP",
              "Food and non-alcoholic beverages",
              "Housing, water, electricity, gas and other fuels",
              "Energy"]

eu = eu[eu["coicop"].isin(categories)]
eu.shape

(56353, 5)

| Eurostat name | Short name |
|---|---|
| All-items HICP | All items |
| Food and non-alcoholic beverages | Food |
| Housing, water, electricity, gas and other fuels | Housing |
| Energy | Energy |

**Expected result:** 56,353 rows.

## Step 3: Keep only the 27 EU countries and the euro area (20 countries)

In [24]:
eu27 = ["Austria", "Belgium", "Bulgaria", "Croatia", "Cyprus", "Czechia",
        "Denmark", "Estonia", "Finland", "France", "Germany", "Greece",
        "Hungary", "Ireland", "Italy", "Latvia", "Lithuania", "Luxembourg",
        "Malta", "Netherlands", "Poland", "Portugal", "Romania", "Slovakia",
        "Slovenia", "Spain", "Sweden"]

eu = eu[eu["geo"].isin(eu27) | eu["geo"].str.contains("20 countries")]
eu.shape

(38413, 5)

The file covers 45 places, including non-EU countries (UK, US, Türkiye…) and several country groups. Only these are kept:
- the **27 EU countries**, to show where Ireland sits among its neighbours
- the **euro area (20 countries)**, as the European average

There are 3 versions of the euro area in the file. The 20-country one is used because it contains the **same countries in every month**, so it's a fair comparison. The others change membership over time.

**Expected result:** 38,413 rows, 28 places.

In [25]:
eu["geo"].nunique()

28

## Step 4: Keep only January 2019 – December 2025

In [26]:
eu = eu[(eu["TIME_PERIOD"] >= "2019-01") & (eu["TIME_PERIOD"] <= "2025-12")]
eu.shape

(9408, 5)

The same date range as the CSO data. `TIME_PERIOD` is text like `2019-01`, and because it's written year first, text comparison puts it in the right order.

## Step 5: Check the flags

In [27]:
eu["OBS_FLAG"].value_counts()

OBS_FLAG
u    8
Name: count, dtype: int64

In [28]:
eu[eu["OBS_FLAG"] == "u"]

,coicop,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG
70387,Food and non-alcoholic beverages,Hungary,2020-04,8.9,u
70388,Food and non-alcoholic beverages,Hungary,2020-05,8.6,u
70389,Food and non-alcoholic beverages,Hungary,2020-06,8.0,u
70399,Food and non-alcoholic beverages,Hungary,2021-04,1.7,u
70400,Food and non-alcoholic beverages,Hungary,2021-05,1.7,u
70401,Food and non-alcoholic beverages,Hungary,2021-06,2.4,u
74577,Food and non-alcoholic beverages,Portugal,2020-04,3.8,u
74589,Food and non-alcoholic beverages,Portugal,2021-04,-0.8,u


In [29]:
eu = eu.drop(columns=["OBS_FLAG"])

`OBS_FLAG` holds notes about some values. After filtering, only **8 values** are flagged `u` (low reliability): Hungary and Portugal, Food, in April–June 2020 and 2021. These are COVID lockdown months, when shops were closed and prices were harder to measure.

**Decision:** keep these values. They are official figures, there are only 8 out of 9,408, and they are not Ireland. The flag column is then dropped because it's empty for every other row.

## Step 6: Rename the columns to match the CSO table

In [30]:
eu = eu.rename(columns={
    "TIME_PERIOD": "month",
    "coicop": "category_full",
    "OBS_VALUE": "value"
})
eu.head()

,category_full,geo,month,value
49187,All-items HICP,Austria,2019-01,1.7
49188,All-items HICP,Austria,2019-02,1.4
49189,All-items HICP,Austria,2019-03,1.7
49190,All-items HICP,Austria,2019-04,1.7
49191,All-items HICP,Austria,2019-05,1.7


| Old name | New name |
|---|---|
| `TIME_PERIOD` | `month` |
| `coicop` | `category_full` |
| `OBS_VALUE` | `value` |
| `geo` | `geo` (unchanged) |

Both tables need the **same column names** so they can be joined later.

## Step 7: Convert the month into a real date

In [31]:
eu["month"] = pd.to_datetime(eu["month"], format="%Y-%m")
eu.dtypes

category_full               str
geo                         str
month            datetime64[us]
value                   float64
dtype: object

`2019-01` is text. Converting it to a real date (`2019-01-01`) puts it in the **same format as the CSO dates**, which is needed to match the two tables. `format="%Y-%m"` means: 4-digit year, a dash, 2-digit month.

## Step 8: Add short names and codes, and shorten the euro area name

In [32]:
eu["category"] = eu["category_full"].map({
    "All-items HICP": "All items",
    "Food and non-alcoholic beverages": "Food",
    "Housing, water, electricity, gas and other fuels": "Housing",
    "Energy": "Energy"
})

eu["coicop_code"] = eu["category"].map({
    "All items": "00",
    "Food": "01",
    "Housing": "04",
    "Energy": "ZZZ3"
})

eu.loc[eu["geo"].str.contains("20 countries"), "geo"] = "Euro area"
eu = eu.drop(columns=["category_full"])
eu.head()

,geo,month,value,category,coicop_code
49187,Austria,2019-01-01,1.7,All items,00
49188,Austria,2019-02-01,1.4,All items,00
49189,Austria,2019-03-01,1.7,All items,00
49190,Austria,2019-04-01,1.7,All items,00
49191,Austria,2019-05-01,1.7,All items,00


- **Short category names and COICOP codes:** Eurostat only has long names, while the CSO table has codes. Adding the **same short names and codes** as the CSO table is what makes the two sources match.
- **Euro area:** the name "Euro area – 20 countries (2023-2025)" is too long for a chart, so it becomes "Euro area".
- **`category_full` removed:** the short name and code carry the same information.

## Step 9: Final check

In [33]:
eu = eu[["month", "geo", "coicop_code", "category", "value"]]
eu.shape

(9408, 5)

In [34]:
eu.isna().sum()

month          0
geo            0
coicop_code    0
category       0
value          0
dtype: int64

In [35]:
eu.groupby("category").size()

category
All items    2352
Energy       2352
Food         2352
Housing      2352
dtype: int64

In [36]:
eu["geo"].unique()

<StringArray>
[    'Austria',     'Belgium',    'Bulgaria',      'Cyprus',     'Czechia',
     'Germany',     'Denmark',   'Euro area',     'Estonia',      'Greece',
       'Spain',     'Finland',      'France',     'Croatia',     'Hungary',
     'Ireland',       'Italy',   'Lithuania',  'Luxembourg',      'Latvia',
       'Malta', 'Netherlands',      'Poland',    'Portugal',     'Romania',
      'Sweden',    'Slovenia',    'Slovakia']
Length: 28, dtype: str

- 9,408 rows × 5 columns, in the same order as the CSO table
- 0 missing values
- 2,352 rows per category (28 places × 84 months)
- 28 places, including "Euro area"

### Summary of Eurostat cleaning
| Step | Rows after |
|---|---|
| Raw file (5 columns loaded) | 3,533,048 |
| 4 categories | 56,353 |
| 27 EU countries + euro area | 38,413 |
| Jan 2019 – Dec 2025 | 9,408 |

**Final table:** 9,408 rows × 5 columns (`month`, `geo`, `coicop_code`, `category`, `value`), no missing values, same columns as the CSO table.

## 1.4 Merging the two datasets

Both cleaned tables now have the same 5 columns: `month`, `geo`, `coicop_code`, `category`, `value`.
Ireland appears in **both** tables, so before combining them we check that the two sources agree.

## Step 1: Line up Ireland from both sources

In [37]:
ireland_eu = eu[eu["geo"] == "Ireland"]

check = cso.merge(ireland_eu,
                  on=["month", "geo", "coicop_code", "category"],
                  suffixes=("_cso", "_eurostat"))
check.shape

(336, 6)

In [38]:
check.head()

,month,geo,coicop_code,category,value_cso,value_eurostat
0,2019-01-01,Ireland,00,All items,0.8,0.8
1,2019-01-01,Ireland,01,Food,-1.6,-1.6
2,2019-01-01,Ireland,04,Housing,5.1,5.1
3,2019-01-01,Ireland,ZZZ3,Energy,1.6,1.6
4,2019-02-01,Ireland,00,All items,0.7,0.7


`merge` joins the CSO and Eurostat rows for Ireland that have the same month, country, code and category, putting the two values side by side. The `suffixes` label which value came from where.

## Step 2: Do the numbers agree?

In [39]:
check["diff"] = check["value_cso"] - check["value_eurostat"]
(check["diff"] != 0).sum()

np.int64(20)

In [40]:
check[check["diff"] != 0]

,month,geo,coicop_code,category,value_cso,value_eurostat,diff
47,2019-12-01,Ireland,ZZZ3,Energy,1.2,1.1,0.1
139,2021-11-01,Ireland,ZZZ3,Energy,28.0,28.1,-0.1
179,2022-09-01,Ireland,ZZZ3,Energy,38.0,38.1,-0.1
187,2022-11-01,Ireland,ZZZ3,Energy,43.1,43.0,0.1
191,2022-12-01,Ireland,ZZZ3,Energy,34.3,34.4,-0.1
195,2023-01-01,Ireland,ZZZ3,Energy,33.1,33.2,-0.1
199,2023-02-01,Ireland,ZZZ3,Energy,29.2,29.3,-0.1
203,2023-03-01,Ireland,ZZZ3,Energy,11.6,11.7,-0.1
207,2023-04-01,Ireland,ZZZ3,Energy,12.1,12.2,-0.1
215,2023-06-01,Ireland,ZZZ3,Energy,-3.1,-3.0,-0.1


In [41]:
check["diff"].abs().max()

np.float64(0.10000000000000142)

A `diff` column is added (CSO value minus Eurostat value), then we count how many rows are different.

**Finding:** 316 of 336 values match exactly. **20 values differ, all by only 0.1, and all in Energy.**

**Why:** the CSO's own methodology note says the rates of change are calculated from index numbers that are **rounded to one decimal place**. Energy is a combined group (electricity, gas, fuels…), so it is built from several rounded numbers, and the two offices end up rounding the last decimal differently. A 0.1 difference on values like 43% is too small to see on a chart, so the sources agree.

## Step 3: Combine the two tables

In [42]:
eu_no_ireland = eu[eu["geo"] != "Ireland"]

data = pd.concat([cso, eu_no_ireland], ignore_index=True)
data.shape

(9408, 5)

Ireland is in both tables, so it is removed from the Eurostat table first; otherwise every Irish month would appear twice. The CSO figures are kept for Ireland because they are the national source, and Step 2 showed they match Eurostat to within 0.1.

`pd.concat` stacks the two tables on top of each other. This works because they have the same 5 columns.

**Expected result:** 9,408 rows × 5 columns (336 Irish rows from CSO + 9,072 rows for the other 27 places from Eurostat).

## 1.5 Saving the clean data

### Step 1: Sort the rows


In [43]:
data = data.sort_values(["geo", "category", "month"]).reset_index(drop=True)
data.head(10)

,month,geo,coicop_code,category,value
0,2019-01-01,Austria,00,All items,1.7
1,2019-02-01,Austria,00,All items,1.4
2,2019-03-01,Austria,00,All items,1.7
3,2019-04-01,Austria,00,All items,1.7
4,2019-05-01,Austria,00,All items,1.7
5,2019-06-01,Austria,00,All items,1.6
6,2019-07-01,Austria,00,All items,1.4
7,2019-08-01,Austria,00,All items,1.5
8,2019-09-01,Austria,00,All items,1.2
9,2019-10-01,Austria,00,All items,1.0


The rows are sorted by country, then category, then month. A sorted file is easier to read and check by eye, and each country's line is already in time order for the chart.

## Step 2: Save as CSV

In [44]:
data.to_csv("data/clean_data.csv", index=False)

The table is saved to `data/clean_data.csv`. This file is the **handover between the two notebooks**: `1_data.ipynb` ends here, and `2_chart.ipynb` starts by loading it.

`index=False` stops pandas adding an extra column of row numbers (0, 1, 2…) that has no meaning.

The clean file is small, so unlike the raw files, it **is** uploaded to GitHub, which means Priyanka gets it automatically.

## Step 3: Reload the file to check it saved correctly

In [45]:
check_file = pd.read_csv("data/clean_data.csv",
                         parse_dates=["month"],
                         dtype={"coicop_code": str})
check_file.shape

(9408, 5)

In [46]:
check_file.dtypes

month          datetime64[us]
geo                       str
coicop_code               str
category                  str
value                 float64
dtype: object

In [47]:
check_file["coicop_code"].unique()

<StringArray>
['00', 'ZZZ3', '01', '04']
Length: 4, dtype: str

Saving to CSV turns everything into plain text, so two things must be told back to pandas when loading:
- `parse_dates=["month"]`: turns `month` back into real dates
- `dtype={"coicop_code": str}`: keeps codes as text. **Without this, `00`, `01` and `04` would turn into the numbers `0`, `1` and `4`** and lose their leading zero.

## 1.6 Key numbers used in the report
Peak annual price rise for Ireland and the euro area, and Ireland's rank among the 27 EU countries (1 = highest peak).

In [48]:
peaks = data.groupby(["category", "geo"])["value"].max().unstack(0)
peaks.loc[["Ireland", "Euro area"]]

category,All items,Energy,Food,Housing
geo,,,,
Ireland,9.6,54.1,13.2,31.8
Euro area,10.6,44.1,17.5,23.2


In [49]:
countries = data[data["geo"] != "Euro area"]
ranks = countries.groupby(["category", "geo"])["value"].max().reset_index()
ranks["rank"] = ranks.groupby("category")["value"].rank(ascending=False, method="min")
ranks[ranks["geo"] == "Ireland"]

,category,geo,value,rank
13,All items,Ireland,9.6,24.0
40,Energy,Ireland,54.1,11.0
67,Food,Ireland,13.2,26.0
94,Housing,Ireland,31.8,11.0
